# 🔗 Bloque 3 — Sesión 2: Cramér's V, significación estadística y selección de variables

**Módulo:** Depuración, Limpieza y Clasificación de Datos (5109)  
**Unidad Didáctica:** UD2 — Verificación estadística de la calidad de los datos  
**Duración estimada:** ~4 horas

---

### Contenidos de esta sesión
- **3.4** Cramér's V: asociación entre variables categóricas
- **3.5** Significación estadística: p-valor, nivel de confianza e intervalo de confianza
- **3.6** Selección de variables: matrices de correlación, multicolinealidad y criterios de descarte

### Criterios de evaluación asociados
> **(CE a)** Se han descrito las técnicas estadísticas de análisis de la calidad técnica de los datos.  
> **(CE b)** Se han aplicado técnicas de evaluación basadas en cálculos estadísticos para detectar la cobertura y el sesgo.  
> **(CE c)** Se ha documentado la descripción del resultado de las verificaciones para que se tomen decisiones de diseño.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, chi2_contingency

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

titanic = sns.load_dataset('titanic')
diamonds = sns.load_dataset('diamonds')
penguins = sns.load_dataset('penguins').dropna()
mpg = sns.load_dataset('mpg')

print('✅ Librerías y datasets cargados')

---
## 3.4 Cramér's V: asociación entre variables categóricas

Pearson y Spearman miden relaciones entre variables numéricas. Para cuantificar la **asociación entre dos variables categóricas** necesitamos **Cramér's V**.

### Fundamento: el estadístico chi-cuadrado (χ²)

Cramér's V se basa en el estadístico chi-cuadrado, que compara la tabla de contingencia observada con la que esperaríamos si las dos variables fueran independientes:

$$\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

- $O_{ij}$: frecuencia **observada** en la celda (fila $i$, columna $j$) de la tabla de contingencia
- $E_{ij}$: frecuencia **esperada** en esa celda si no hubiera asociación = (total fila $i$ × total columna $j$) / $n$

> ℹ️ El test chi-cuadrado como contraste de hipótesis se estudia en detalle en el **Bloque 4, Sesión 1**. Aquí usamos su estadístico como ingrediente de V y, más abajo, adelantamos solo sus hipótesis para poder leer el p-valor.

El valor de χ² crece con el tamaño de la muestra y de la tabla, así que no sirve para comparar. Cramér's V lo normaliza para obtener un valor entre 0 y 1:

$$V = \sqrt{\frac{\chi^2 / n}{\min(k-1,\, r-1)}}$$

donde $n$ es el número total de observaciones, $r$ el número de filas y $k$ el número de columnas de la tabla.

| Valor de V | Interpretación |
|-----------|---------------|
| 0.00 – 0.10 | Asociación muy débil o nula |
| 0.10 – 0.30 | Asociación débil |
| 0.30 – 0.50 | Asociación moderada |
| 0.50 – 1.00 | Asociación fuerte |

### ¿Qué hipótesis se contrastan? El p-valor que acompaña a V

Cramér's V **no es un test**: es una medida del tamaño de la asociación (un *tamaño del efecto*). El contraste de hipótesis lo hace el **test chi-cuadrado de independencia** del que procede, y el p-valor que devuelve nuestra función `cramers_v()` es el de ese test. Sus hipótesis son:

- **H₀ (hipótesis nula):** las dos variables son **independientes** en la población. Conocer la categoría de una no dice nada sobre la otra. Equivale a decir que la V poblacional es 0.
- **H₁ (hipótesis alternativa):** las dos variables **están asociadas** (no son independientes). Equivale a decir que la V poblacional es mayor que 0.

Las frecuencias esperadas $E_{ij}$ son precisamente las que veríamos **si H₀ fuera cierta**; por eso χ² (y con él V) crece cuanto más se aleja la tabla observada de la independencia. Bajo H₀, el estadístico sigue una distribución χ² con $(r-1)(k-1)$ grados de libertad, y de ahí sale el p-valor.

| Resultado | Decisión | Lectura |
|-----------|----------|---------|
| p-valor < α (0,05) | Se rechaza H₀ | Hay evidencia de asociación. **Cuánta** asociación lo dice V, no el p-valor. |
| p-valor ≥ α | No se rechaza H₀ | No hay evidencia suficiente de asociación (no demuestra que sean independientes). |

Ejemplo con `sex` y `survived`: H₀ = «la supervivencia no depende del sexo»; H₁ = «la supervivencia está asociada al sexo».

> 💡 El contraste es siempre **unilateral** (V no puede ser negativa) y H₁ no indica *qué* categorías se asocian con cuáles: para eso hay que mirar la tabla de contingencia. El significado general del p-valor y de α se desarrolla en el apartado 3.5.

> ⚠️ **Condiciones de uso:**  
> – V indica la **fuerza** de la asociación, pero no su dirección (no tiene signo).  
> – El χ² es fiable cuando las frecuencias esperadas son suficientemente grandes (regla habitual: $E_{ij} \geq 5$ en todas las celdas). Con categorías muy poco frecuentes conviene agruparlas antes.  
> – En tablas 2×2, `chi2_contingency` aplica por defecto la corrección de continuidad de Yates, que reduce ligeramente el χ². Para que V coincida con la fórmula anterior usamos `correction=False`.

In [ ]:
def cramers_v(col1, col2):
    """Calcula el coeficiente V de Cramér entre dos variables categóricas."""
    tabla = pd.crosstab(col1, col2)
    chi2, p, dof, _ = chi2_contingency(tabla, correction=False)
    n = tabla.sum().sum()
    v = np.sqrt(chi2 / (n * (min(tabla.shape) - 1)))
    return v, p

pares = [
    ('sex', 'survived'),
    ('pclass', 'survived'),
    ('embarked', 'survived'),
    ('sex', 'pclass'),
    ('embarked', 'pclass'),
]

print(f"{'Par de variables':<25} {'Cramér V':>10} {'p-valor':>12}   {'Asociación'}")
print('─' * 65)
for c1, c2 in pares:
    df_tmp = titanic[[c1, c2]].dropna()
    V, p = cramers_v(df_tmp[c1], df_tmp[c2])
    if V < 0.10: etiqueta = 'Muy débil'
    elif V < 0.30: etiqueta = 'Débil'
    elif V < 0.50: etiqueta = 'Moderada'
    else: etiqueta = 'Fuerte'
    par = f"{c1} vs {c2}"
    print(f"{par:<25} {V:>10.4f} {p:>12.2e}   {etiqueta}")

> 🔎 **Qué observar:** el sexo es la variable más asociada a la supervivencia (V ≈ 0,54, fuerte), seguida de la clase (moderada). Todos los p-valores son muy pequeños, también los de asociaciones débiles como `sex` vs `pclass` (V ≈ 0,14): con casi 900 pasajeros, casi cualquier asociación resulta «significativa». Por eso hay que mirar **siempre V además del p-valor** (lo desarrollamos en 3.5).

In [ ]:
df_sv = titanic[['sex', 'survived']].dropna()   # eliminamos nulos de las dos columnas a la vez
tabla_sv = pd.crosstab(df_sv['sex'], df_sv['survived'],
                        rownames=['Sexo'], colnames=['Sobrevivió'])
tabla_sv.columns = ['No', 'Sí']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

tabla_sv.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'],
              edgecolor='white', width=0.6)
axes[0].set_title('Frecuencias absolutas')
axes[0].set_xlabel('Sexo')
axes[0].set_ylabel('Nº pasajeros')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Sobrevivió')

tabla_pct = tabla_sv.div(tabla_sv.sum(axis=1), axis=0)
tabla_pct.plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'],
               edgecolor='white', width=0.6)
axes[1].set_title('Proporciones (por fila)')
axes[1].set_xlabel('Sexo')
axes[1].set_ylabel('Proporción')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Sobrevivió')

V, p = cramers_v(df_sv['sex'], df_sv['survived'])
plt.suptitle(f"Sexo vs. Supervivencia — Cramér V={V:.3f}, p={p:.2e}", fontsize=13)
plt.tight_layout()
plt.show()

> 🔎 **Qué observar:** las frecuencias absolutas engañan porque había muchos más hombres que mujeres a bordo. Las **proporciones por fila** muestran la asociación con claridad: sobrevivió alrededor del 74 % de las mujeres frente al 19 % de los hombres. Esa diferencia entre filas es lo que V resume en un solo número.

In [ ]:
def matriz_cramers_v(df, columnas_cat):
    """Calcula la matriz de Cramér's V para un conjunto de variables categóricas."""
    n = len(columnas_cat)
    matrix = pd.DataFrame(np.zeros((n, n)), index=columnas_cat, columns=columnas_cat)
    for i, c1 in enumerate(columnas_cat):
        for j, c2 in enumerate(columnas_cat):
            if i == j:
                matrix.loc[c1, c2] = 1.0
            elif i < j:
                df_tmp = df[[c1, c2]].dropna()
                V, _ = cramers_v(df_tmp[c1], df_tmp[c2])
                matrix.loc[c1, c2] = V
                matrix.loc[c2, c1] = V
    return matrix

cols_cat = ['sex', 'pclass', 'embarked', 'who', 'class', 'survived']
mv = matriz_cramers_v(titanic, cols_cat)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(mv, annot=True, fmt='.3f', cmap='YlOrRd',
            vmin=0, vmax=1, linewidths=0.5, linecolor='white',
            ax=ax, square=True)
ax.set_title("Matriz de Cramér's V — Variables categóricas Titanic", fontsize=13)
plt.tight_layout()
plt.show()

> 🔎 **Qué observar:**
> - `class` y `pclass` tienen **V = 1,000**: son la misma variable codificada de dos formas (texto y número). No es un hallazgo, es una **duplicidad** del dataset.
> - `who` y `sex` tienen V ≈ 0,95: `who` (man / woman / child) se construye a partir del sexo y la edad, así que es casi redundante con `sex`.
>
> 💡 **Implicación para la calidad de datos:** una V cercana a 1 entre dos categóricas es el equivalente a un |r| ≈ 1 entre numéricas. Suele delatar columnas duplicadas o derivadas unas de otras, y solo debería entrar una de ellas en el modelo.

---
## 3.5 Significación estadística: p-valor, nivel de confianza e intervalo de confianza

### El p-valor

El **p-valor** responde a: *si en la población real no hubiera relación (H₀), ¿cuál sería la probabilidad de observar un resultado al menos tan extremo como el medido?*

- **p-valor < α** → rechazamos H₀ → resultado **estadísticamente significativo**
- **p-valor ≥ α** → no podemos rechazar H₀

El umbral habitual es **α = 0.05** (nivel de significación). Su complementario, **1 − α = 95 %**, es el **nivel de confianza**: si H₀ fuera cierta y repitiéramos el estudio muchas veces, solo en el 5 % de las ocasiones la rechazaríamos por error.

> ⚠️ **Errores frecuentes:**  
> – El p-valor NO es la probabilidad de que H₀ sea verdadera.  
> – Significativo ≠ importante. Con n grande, correlaciones minúsculas pueden ser «significativas».  
> – No significativo ≠ sin efecto.

In [ ]:
cols_num = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
print('Correlaciones de Pearson con p-valor (Penguins):\n')
print(f"{'Par':<45} {'r':>8} {'p-valor':>12} {'Significativa (α=0.05)?':>25}")
print('─' * 93)

for i, c1 in enumerate(cols_num):
    for j, c2 in enumerate(cols_num):
        if j <= i: continue
        r, p = pearsonr(penguins[c1], penguins[c2])
        sig = '✅ Sí' if p < 0.05 else '❌ No'
        par = f"{c1} vs {c2}"
        print(f"{par:<45} {r:>8.4f} {p:>12.2e} {sig:>25}")

> 🔎 **Qué observar:** las seis correlaciones son significativas, incluida la más débil (`bill_length_mm` vs `bill_depth_mm`, r = −0,23). Recuerda de la Sesión 1 que esa correlación negativa era un efecto de mezclar especies (paradoja de Simpson): **un p-valor pequeño no garantiza que la correlación signifique lo que parece**, solo que es improbable que sea fruto del azar del muestreo.

In [ ]:
# Misma correlación observada (r = 0.15), distintos tamaños de muestra
# El p-valor de Pearson solo depende de r y de n:  t = r·√(n−2) / √(1−r²)
r_fijo = 0.15
resultados = []
for n in [20, 50, 100, 200, 500, 1000, 5000]:
    t = r_fijo * np.sqrt(n - 2) / np.sqrt(1 - r_fijo**2)
    p = 2 * stats.t.sf(abs(t), df=n - 2)
    resultados.append({'n': n, 'r': r_fijo, 'p-valor': f'{p:.2e}',
                       'Significativa (α=0.05)': '✅ Sí' if p < 0.05 else '❌ No'})

df_res = pd.DataFrame(resultados).set_index('n')
print(df_res.to_string())

# Tamaño mínimo de muestra para que r = 0.15 sea significativa
n_min = next(n for n in range(4, 10000)
             if 2 * stats.t.sf(r_fijo * np.sqrt(n - 2) / np.sqrt(1 - r_fijo**2), df=n - 2) < 0.05)
print()
print(f'→ Exactamente la misma correlación (r = {r_fijo}) pasa de no significativa a muy significativa')
print(f'  solo por aumentar n. A partir de n = {n_min} ya supera el umbral de α = 0.05.')


> 🔎 **Qué observar:** el coeficiente no cambia (r = 0,15, una relación débil que explica alrededor del 2 % de la variabilidad), pero el p-valor pasa de 0,5 a prácticamente 0. **El p-valor mide cuánta evidencia hay de que la relación no es cero, no cuánto de fuerte es.** En datasets grandes casi todo es significativo, así que la decisión debe apoyarse en el tamaño del coeficiente.

### Intervalo de confianza para la correlación

El r que calculamos es una estimación a partir de una muestra; con otra muestra saldría un valor algo distinto. El **intervalo de confianza** (IC) es un rango de valores plausibles para la correlación real de la población.

> ⚠️ **Interpretación correcta:** un IC al 95 % **no** significa que «el valor real está dentro con un 95 % de probabilidad». Significa que el *procedimiento* acierta el 95 % de las veces: si repitiéramos el muestreo muchas veces y calculáramos un IC cada vez, el 95 % de esos intervalos contendría el valor real.

Para la correlación de Pearson se usa la **transformación z de Fisher**:

$$z = \frac{1}{2} \ln\left(\frac{1+r}{1-r}\right) = \text{arctanh}(r), \quad SE_z = \frac{1}{\sqrt{n-3}}$$

1. Se transforma r a z.
2. Se calcula el intervalo en la escala z: $z \pm 1{,}96 \cdot SE_z$ (para el 95 %).
3. Se deshace la transformación en los dos extremos: $r = \tanh(z)$.

**Relación con el p-valor:** si el IC al 95 % **no contiene el 0**, la correlación es significativa con α = 0,05.

In [ ]:
def ic_pearson(r, n, alpha=0.05):
    """IC para la correlación de Pearson mediante la z de Fisher."""
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    z_crit = stats.norm.ppf(1 - alpha / 2)
    z_lo = z - z_crit * se
    z_hi = z + z_crit * se
    return np.tanh(z_lo), np.tanh(z_hi)

r_obs = 0.60
print(f'IC al 95% para r = {r_obs}:\n')
print(f"{'n':>6}  {'IC inferior':>12}  {'IC superior':>12}  {'Amplitud':>10}")
print('─' * 45)
for n in [20, 50, 100, 300, 1000]:
    lo, hi = ic_pearson(r_obs, n)
    print(f'{n:>6}  {lo:>12.4f}  {hi:>12.4f}  {hi-lo:>10.4f}')
print()
print('→ Con n=20, el IC es amplísimo: el r real podría ser débil o muy fuerte. Con n=1000, el IC es preciso.')

In [ ]:
n = 100
r_vals = np.linspace(-0.9, 0.9, 37)
fig, ax = plt.subplots(figsize=(11, 5))

for r_v in r_vals:
    lo, hi = ic_pearson(r_v, n)
    color = 'steelblue' if r_v >= 0 else 'coral'
    ax.plot([r_v, r_v], [lo, hi], color=color, linewidth=1.5, alpha=0.7)
    ax.scatter([r_v], [r_v], color=color, s=20, zorder=3)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('r observado')
ax.set_ylabel('Intervalo de confianza (95%)')
ax.set_title(f'IC al 95% para la correlación de Pearson (n={n})', fontsize=13)
plt.tight_layout()
plt.show()

> 🔎 **Qué observar:**
> - Los intervalos que **cruzan la línea horizontal del 0** corresponden a correlaciones no significativas. Con n = 100 eso ocurre, aproximadamente, cuando |r| < 0,20.
> - Los intervalos son **más estrechos cuanto más cerca está r de ±1** y son asimétricos (r no puede pasar de 1).
>
> 💡 `scipy` también lo calcula directamente: `pearsonr(x, y).confidence_interval(0.95)`.

---
## 3.6 Selección de variables mediante correlación

A la hora de preparar los datos para un modelo, la correlación ayuda a responder dos preguntas distintas:

1. **Relevancia:** ¿qué predictores están relacionados con la variable **objetivo**? Aquí una correlación alta es *buena*.
2. **Redundancia:** ¿qué predictores están muy relacionados **entre sí**? Aquí una correlación alta es *un problema*.

### Multicolinealidad

Cuando dos o más variables **predictoras** están altamente correlacionadas entre sí, tenemos **multicolinealidad**: aportan casi la misma información. Causa coeficientes inestables en regresión lineal/logística y dificulta la interpretación.

> ⚠️ La multicolinealidad se refiere solo a los predictores. Una correlación alta entre un predictor y el objetivo no es multicolinealidad: es justo lo que buscamos.

**Criterio habitual (orientativo):**
- |r| > 0.70 entre dos predictores → correlación alta, conviene vigilarla
- |r| > 0.85 entre dos predictores → redundancia fuerte, considerar eliminar uno (normalmente el menos relacionado con el objetivo o el más difícil de obtener)

> ℹ️ La matriz de correlación solo detecta redundancia **por pares**. Una variable puede ser combinación de varias otras sin que ningún par destaque; para eso existe el VIF (*Variance Inflation Factor*), que no trataremos aquí.

In [ ]:
# Ejemplo: dataset mpg (consumo de coches). Objetivo: mpg (millas por galón)
# Buscamos redundancia SOLO entre los predictores
cols_pred = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']
r_mat = mpg[cols_pred].corr()

print('Pares de predictores con correlación alta (|r| > 0.70):')
print()
for i, c1 in enumerate(cols_pred):
    for j, c2 in enumerate(cols_pred):
        if j <= i: continue
        r = r_mat.loc[c1, c2]
        if abs(r) > 0.70:
            nivel = '⚠️ Redundancia fuerte (> 0.85)' if abs(r) > 0.85 else '〰️ Correlación alta'
            print(f'  {c1:<13} vs {c2:<13} r = {r:+.3f}  {nivel}')

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(r_mat, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white', ax=ax, square=True)
ax.set_title('Correlación entre predictores — mpg', fontsize=12)
plt.tight_layout()
plt.show()


> 🔎 **Qué observar:** `cylinders`, `displacement`, `horsepower` y `weight` forman un bloque de variables muy correlacionadas entre sí: todas miden, en el fondo, «lo grande que es el motor/coche». Meter las cuatro en una regresión aportaría poca información nueva y haría inestables los coeficientes. Lo razonable es quedarse con una o dos representantes del bloque.

In [ ]:
def informe_seleccion_variables(df, cols_predictoras, col_objetivo=None,
                                 umbral_multicolinealidad=0.85,
                                 umbral_relevancia=0.10):
    """Informe de selección de variables por correlación de Pearson.

    Solo admite columnas numéricas. Recuerda que Pearson mide relación LINEAL:
    una variable con relación no lineal con el objetivo puede aparecer como 'Baja'.
    """
    todas = cols_predictoras + ([col_objetivo] if col_objetivo else [])
    no_numericas = [c for c in todas if not pd.api.types.is_numeric_dtype(df[c])]
    if no_numericas:
        raise TypeError(f'Columnas no numéricas (codifícalas o exclúyelas): {no_numericas}')
    datos = df[todas].dropna()

    print('=' * 65)
    print('INFORME DE SELECCIÓN DE VARIABLES')
    print('=' * 65)

    if col_objetivo:
        print(f"\n1. CORRELACIÓN CON '{col_objetivo}':")
        print(f'   (umbral mínimo: |r| > {umbral_relevancia})')
        print()
        relevancia = []
        for col in cols_predictoras:
            r, p = pearsonr(datos[col], datos[col_objetivo])
            relevante = '✅' if abs(r) > umbral_relevancia and p < 0.05 else '❌ Baja'
            relevancia.append((col, r, p, relevante))
            print(f'   {col:<25} r={r:+.3f}  p={p:.2e}  {relevante}')

    print(f'\n2. MULTICOLINEALIDAD (umbral: |r| > {umbral_multicolinealidad}):')
    r_pred = datos[cols_predictoras].corr()
    hay_multi = False
    for i, c1 in enumerate(cols_predictoras):
        for j, c2 in enumerate(cols_predictoras):
            if j <= i: continue
            r = r_pred.loc[c1, c2]
            if abs(r) > umbral_multicolinealidad:
                print(f'   ⚠️  {c1} vs {c2}: r = {r:.3f}')
                hay_multi = True
    if not hay_multi:
        print('   ✅ Ningún par de predictores supera el umbral.')

    print('\n' + '=' * 65)

cols_pred_p = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']
informe_seleccion_variables(penguins, cols_pred_p, col_objetivo='body_mass_g')

> 🔎 **Qué observar:** en Penguins los tres predictores son relevantes para estimar la masa corporal y ninguno es redundante con otro. La correlación de 0,87 entre `flipper_length_mm` y `body_mass_g` no aparece en el apartado 2 porque `body_mass_g` es el **objetivo**: es relevancia, no multicolinealidad.

In [ ]:
cols_titanic = ['age', 'fare', 'sibsp', 'parch', 'pclass']
objetivo = 'survived'
df_t = titanic[cols_titanic + [objetivo]].dropna().copy()
df_t['pclass'] = df_t['pclass'].astype(float)

r_obj = df_t[cols_titanic].corrwith(df_t[objetivo])
r_pred = df_t[cols_titanic].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colores = ['steelblue' if abs(v) > 0.1 else 'lightgray' for v in r_obj]
axes[0].barh(r_obj.index, r_obj.values, color=colores, edgecolor='white')
axes[0].axvline(0.1, color='crimson', linestyle='--', linewidth=1.5, label='Umbral +0.1')
axes[0].axvline(-0.1, color='crimson', linestyle='--', linewidth=1.5, label='Umbral -0.1')
axes[0].set_title(f"Correlación con '{objetivo}'", fontsize=12)
axes[0].set_xlabel('r de Pearson')
axes[0].legend(fontsize=9)

sns.heatmap(r_pred, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white',
            ax=axes[1], square=True)
axes[1].set_title('Multicolinealidad entre predictores', fontsize=12)

plt.suptitle('Análisis de selección de variables — Titanic', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
informe_seleccion_variables(
    df_t, cols_titanic, col_objetivo='survived',
    umbral_multicolinealidad=0.85, umbral_relevancia=0.10
)

### ⚠️ Límite del filtro por correlación: relaciones no lineales

Según el informe, `age` tiene relevancia «Baja» (r = −0,08) y la descartaríamos. Antes de hacerlo, miremos la tasa de supervivencia por tramos de edad.

In [ ]:
# Tasa de supervivencia por tramos de edad
tramos = pd.cut(df_t['age'], bins=[0, 12, 18, 30, 45, 60, 80],
                labels=['0-12', '13-18', '19-30', '31-45', '46-60', '61-80'])
tasa = df_t.groupby(tramos, observed=True)['survived'].agg(['mean', 'count'])

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(tasa.index.astype(str), tasa['mean'], color='steelblue', edgecolor='white')
ax.axhline(df_t['survived'].mean(), color='crimson', linestyle='--', linewidth=1.5, label='Media global')
for i, (m, c) in enumerate(zip(tasa['mean'], tasa['count'])):
    ax.text(i, m + 0.01, f'{m:.0%}\n(n={c})', ha='center', fontsize=9)
ax.set_ylim(0, 0.75)
ax.set_xlabel('Edad')
ax.set_ylabel('Tasa de supervivencia')
ax.set_title("Supervivencia por tramos de edad — r de Pearson con 'age' = "
             f"{df_t['age'].corr(df_t['survived']):+.2f}")
ax.legend()
plt.tight_layout()
plt.show()


> 🔎 **Qué observar:** los niños sobrevivieron claramente por encima de la media y los mayores de 60 por debajo, pero entre los 13 y los 60 años la tasa apenas cambia. La relación existe, pero **no es lineal**, y Pearson la resume en un r casi nulo. Descartar `age` por ese r sería un error.
>
> 💡 **Conclusiones para la selección de variables:**
> - El filtro por correlación lineal es un **primer cribado**, no una decisión definitiva. Antes de descartar una variable, visualízala frente al objetivo.
> - Con objetivo binario (como `survived`) el r de Pearson sigue siendo calculable (se llama correlación *biserial puntual*), pero es poco sensible. `pclass`, además, es ordinal: Spearman sería más apropiado.
> - Para captar relaciones no lineales veremos la **información mutua** en el Bloque 4, Sesión 2.

---
## 🔧 Ejercicios

### Ejercicio 1 — Cramér's V en Diamonds

Calcula la matriz de Cramér's V entre `cut`, `color` y `clarity` y represéntala con un heatmap.
1. ¿Qué par está más asociado? ¿Cómo calificarías esa asociación según la tabla de interpretación?
2. Calcula el p-valor de cada par con `cramers_v()`. ¿Son significativas las asociaciones?
3. Explica la aparente contradicción entre las respuestas 1 y 2 (pista: ¿cuántas filas tiene `diamonds`?).

In [ ]:
# 1. Matriz de Cramér's V y heatmap
# cols_cat_d = [...]
# mv_d = matriz_cramers_v(...)

# 2. p-valor de cada par
# V, p = cramers_v(diamonds[...], diamonds[...])

# 3. Tamaño del dataset
# ...


### Ejercicio 2 — IC de correlación

Para `flipper_length_mm` vs `body_mass_g` del dataset `penguins`:
1. Calcula r y su IC al 95 % con todos los datos.
2. Repite el cálculo con una **muestra aleatoria** de 20 pingüinos (`.sample(20, random_state=42)`). ¿Cómo cambia el IC? ¿Por qué?
3. Repítelo ahora con las **primeras 20 filas** (`.head(20)`). El r cambia mucho más que en el apartado 2. ¿Por qué? (pista: mira qué especies hay en esas 20 filas y recuerda la paradoja de Simpson de la Sesión 1). ¿Qué te dice esto sobre cómo debe tomarse una muestra?

In [ ]:
# 1. Todos los datos
# peng_full = penguins[['flipper_length_mm', 'body_mass_g', 'species']]
# r_full, _ = pearsonr(...)
# lo_full, hi_full = ic_pearson(r_full, len(peng_full))

# 2. Muestra aleatoria de 20
# ...

# 3. Primeras 20 filas
# ...


### Ejercicio 3 — Informe de selección completo

Aplica `informe_seleccion_variables()` al dataset `diamonds` con las variables numéricas `carat`, `depth`, `table`, `x`, `y`, `z` y objetivo `price` (usa un umbral de relevancia de 0.2).
1. ¿Qué variables descartarías por baja relevancia? Antes de decidir, dibuja un scatter de cada una frente a `price` para comprobar que no hay una relación no lineal.
2. ¿Qué pares de predictores son redundantes? ¿Por qué tiene sentido que lo sean?
3. Propón un subconjunto final de variables para predecir el precio y justifícalo.

In [ ]:
# 1. Informe
# informe_seleccion_variables(diamonds, [...], col_objetivo=..., umbral_relevancia=0.2)

# Scatter de las candidatas a descarte frente a price
# ...

# 3. Subconjunto final propuesto y justificación
# ...


---
## 📝 Resumen de la sesión

| Concepto | Clave |
|----------|-------|
| **Cramér's V** | Fuerza de la asociación entre categóricas. Rango [0,1], sin signo. Basado en chi-cuadrado. V ≈ 1 suele delatar columnas duplicadas. |
| **p-valor** | Probabilidad de observar un resultado al menos tan extremo si H₀ fuera cierta. Umbral habitual: α=0.05. |
| **Significativo ≠ importante** | Con n grande, efectos mínimos pueden ser significativos. |
| **IC de Pearson** | Rango plausible para el r real. Se amplía con n pequeño. Si no contiene el 0, r es significativa. |
| **Multicolinealidad** | \|r\| > 0.85 entre **predictores** → redundancia → considerar eliminar uno. |
| **Selección por correlación** | Combinar relevancia con el objetivo + redundancia entre predictores. Es un primer cribado: no detecta relaciones no lineales. |

### 🔜 Próxima sesión
**Bloque 4 — Sesión 1:** Tests de independencia. Chi-cuadrado, t-test y Mann-Whitney.